In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import sys
sys.path.insert(0, '../.')
import wlasl_dataloader


2025-04-08 06:59:16.151443: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-08 06:59:17.736798: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744109958.106403  978415 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744109958.177787  978415 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744109959.070715  978415 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [7]:

class GCNClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_layers=3, num_classes=12):
        """
        in_channels: Dimension C of node features.
        hidden_channels: Hidden dimension for GCN and dense layers.
        num_classes: Number of output classes.
        """
        super(GCNClassifier, self).__init__()
        # Three GCNConv layers
        # self.convs = []
        # for i in range(num_layers):
        #     if i == 0:
        #         self.convs.append(GCNConv(input_dim, hidden_dim))
        #     elif i == 1:
        #         self.convs.append(GCNConv(hidden_dim, hidden_dim*2))
        #     elif i != num_layers-1:
        #         self.convs.append(GCNConv(hidden_dim*2, hidden_dim*2))
        #     else:
        #         self.convs.append(GCNConv(hidden_dim*2, output_dim))
        # self.convs = nn.ModuleList(self.convs)
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels*2)
        self.conv3 = GCNConv(hidden_channels*2, hidden_channels)
        
        # Three dense (fully-connected) layers.
        # After global pooling and temporal aggregation, the input dimension is hidden_channels.
        self.fc1 = nn.Linear(hidden_channels, hidden_channels)
        self.fc2 = nn.Linear(hidden_channels, hidden_channels // 2)
        self.fc3 = nn.Linear(hidden_channels // 2, num_classes)

        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, data):
        """
        x: Tensor of shape [B, T, R, C]
        edge_index: LongTensor of shape [2, num_edges] for a single graph 
                    (assumed to be the same for each frame).
        """
        x, edge_index = data
        print(x.size(), edge_index.size())
        B, T, R, C = x.size()
        device = x.device

        # Reshape to treat each frame as an individual graph.
        # New shape: [B * T, R, C]
        x = x.view(B * T, R, C)
        
        # Prepare the batch vector for global pooling.
        # Each graph (i.e. each frame) is assigned an index from 0 to (B*T - 1).
        batch_graph = torch.arange(B * T, device=device).repeat_interleave(R)
        
        # Flatten node features to shape [B*T * R, C]
        x = x.view(B * T * R, C)
        
        # The provided edge_index is for one graph with R nodes.
        # We need to replicate and offset it for each graph in the batch.
        # For each graph i in 0,...,B*T-1, we add an offset of i*R.
        num_edges = edge_index.size(1)
        new_edge_index = []
        for i in range(B * T):
            offset = i * R
            new_edge_index.append(edge_index + offset)
        new_edge_index = torch.cat(new_edge_index, dim=1)
        
        # Apply three GCNConv layers with ReLU activations.
        x = F.relu(self.conv1(x, new_edge_index))
        x = F.relu(self.conv2(x, new_edge_index))
        x = F.relu(self.conv3(x, new_edge_index))
        
        # Global pooling over nodes for each graph.
        x = global_mean_pool(x, batch_graph)  # Shape: [B*T, hidden_channels]
        
        # Reshape back to separate the T frames per batch sample.
        x = x.view(B, T, -1)
        
        # Aggregate the T frame outputs (for example, take the mean over time).
        x = x.mean(dim=1)  # Now x has shape [B, hidden_channels]
        
        # Pass through the fully-connected (dense) layers.
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # Final output shape: [B, num_classes]
        
        return self.softmax(x)


In [3]:
import yaml
from dotmap import DotMap
import torch

# Example usage:
# Suppose:
# B = 4, T = 5, R = 10, C = 16, hidden_channels = 32
B, T, R, C = 4, 20, 56, 56
hidden_channels = 64

# Random input tensor with shape [B, T, R, C]
x = torch.randn(B, T, R, C)

# Define an example edge_index for a graph with R nodes.
# Here we create a fully-connected directed graph (including self-loops)

with open('../config/asl.yaml', 'r') as f:
    config = yaml.safe_load(f)

config = DotMap(config)

row = []
col = []
for i in range(R):
    for j in range(R):
        row.append(i)
        col.append(j)
edge_index = torch.tensor([row, col], dtype=torch.long)

graphs_path = '/home/tkg5kq/.cache/kagglehub/datasets/risangbaskoro/wlasl-processed/versions/5/graphs/'
label_map = {'abdomen': 0, 'good': 1, 'yes': 2}
sequence, labels = wlasl_dataloader.collect_graph_data(['abdomen', 'good', 'yes'], graphs_path, 56, config, label_map)


In [4]:

from sklearn.model_selection import train_test_split
import random
print(len(sequence), len(labels))
print(type(sequence[0]))
print(len(sequence[0]))
sequence[0]
# tensor_seq = torch.stack(sequence)
# print(tensor_seq.shape, x.shape)
random_sequence = random.sample(range(len(sequence)), int(len(sequence)*.8))

X_train = []
X_test = []
y_train = []
y_test = []
for i in range(len(sequence)):
    if i in random_sequence:
        X_train.append(sequence[i])
        y_train.append(labels[i])
    else:
        X_test.append(sequence[i])
        y_test.append(labels[i])
# X_train, X_test, y_train, y_test = train_test_split(sequence, labels, test_size=config.test_size, random_state=config.seed)
len(X_train), len(X_test), len(y_train), len(y_test)

210 210
<class 'list'>
20


(168, 42, 168, 42)

In [8]:
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
from torch.optim import SGD
from torchvision.transforms.functional import pad
# from torch_geometric.loader import DataLoader

def collate_fn(data):
    # print(type(data))
    # print(len(data))
    d_x = []
    d_edge = []
    max_len = 50
    for d in data:
        x = []
        edge = []
        for element in d:
            x.append(torch.tensor(element.x))
            edge.append(torch.tensor(element.adj_t))
        d_x.append(torch.stack(x))
        d_edge.append(torch.stack(edge))
    d_x = torch.stack(d_x)
    d_edge = torch.stack(d_edge)
    print(d_x.shape, d_edge.shape)
    return d_x, d_edge

train_loader = DataLoader(X_train, batch_size=32, shuffle=True, collate_fn=collate_fn)
# Create the model and forward pass.
model = GCNClassifier(in_channels=C, hidden_channels=hidden_channels, num_classes=3)

optimizer = SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

def accuracy(out, y):
    correct = 0
    total = 0
    for pred, label in zip(out, y):
        total += 1
        correct += (pred == label)
    return correct, total

for epoch in range(config.epochs):
    running_loss = 0
    total_runs = 0
    total_correct = 0
    for x, edge_index in train_loader:
        optimizer.zero_grad()

        # print(X_train)
        # print(data)
        out = model((x, edge_index))
        print(out.shape)
        break
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct, total = accuracy(out.argmax(1), y)
        total_correct += correct
        total_runs += total 

    if (epoch+1) % 2 == 0:
        print(f'Epoch {epoch+1}/{config.epochs}: Loss: {running_loss/total_runs}, Accuracy: {total_correct}/{total_runs}')
    break

/tmp/ipykernel_978415/3341708102.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x.append(torch.tensor(element.x))


AttributeError: 'GlobalStorage' object has no attribute 'adj_t'